# Activity: QuickCart Delivery Analytics
### Applying Histograms, Density Plots & Proportions to a Real Business Question
Unit 1, Lecture 7 — Visualization of Data, Understanding and Interpretation (UE24CS342AA9)

---

## Scenario

You are a data analyst at **QuickCart**, an online delivery startup. The operations team
has asked you two questions ahead of tomorrow's planning meeting:

1. **"What does our delivery time actually look like?** Are we mostly fast, mostly slow,
   or is something more complicated going on?"
2. **"Which region should we prioritize for extra delivery staff?"**

You've been given delivery records for the last month. Your job is to visualize the data
correctly enough to give the ops team a confident answer — using the same tools from
today's lecture: histograms, bin width, density plots, bandwidth, and proportion charts.

Run the cell below to load your dataset — you do not need to change it.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (7, 4.5)

# --- QuickCart delivery dataset (synthetic, seeded for reproducibility) ---
rng = np.random.default_rng(42)
n = 1200

regions = rng.choice(["North", "South", "East", "West", "Central"],
                      size=n, p=[0.22, 0.18, 0.22, 0.18, 0.20])

is_express = rng.random(n) < 0.35
delivery_time = np.where(
    is_express,
    rng.normal(loc=5, scale=1.5, size=n),
    rng.normal(loc=18, scale=4.5, size=n),
)
delivery_time = np.clip(delivery_time, 0.5, None)

south_mask = regions == "South"
delivery_time[south_mask] += rng.normal(loc=8, scale=3, size=south_mask.sum())

df = pd.DataFrame({"region": regions, "delivery_time_hours": delivery_time})

def status(t):
    if t <= 8:
        return "On Time"
    elif t <= 20:
        return "Delayed"
    else:
        return "Severely Delayed"

df["delivery_status"] = df["delivery_time_hours"].apply(status)

print(f"Loaded {len(df)} delivery records")
df.head()

Loaded 1200 delivery records


,region,delivery_time_hours,delivery_status
0,West,14.657945,Delayed
1,East,12.668919,Delayed
2,Central,22.652142,Severely Delayed
3,West,10.331552,Delayed
4,North,3.222223,On Time


## Task 1 — Histogram: What does delivery time look like? 

Fill in the blank below to plot a histogram of `delivery_time_hours`. Try **three different
bin widths** by changing the `bins` value: something small (e.g. `50`), something moderate
(e.g. `20`), and something large (e.g. `4`).

**Fill in the blank marked `# TODO`.**

In [2]:
# Chose 20 bins after trying 50 (too spiky, over-fits noise) and 4 (too coarse, hides
# structure). 20 bins is fine enough to reveal both peaks without fragmenting them.
bins = 20

plt.figure()
plt.hist(df["delivery_time_hours"], bins=bins, color="#4C72B0", edgecolor="white")
plt.title(f"Delivery Time Distribution (bins={bins})")
plt.xlabel("Delivery Time (hours)")
plt.ylabel("Number of Orders")
plt.show()


C:\Users\moksh\AppData\Local\Temp\ipykernel_11272\1848355582.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Question 1:** At a bin width that shows the clearest picture, does the data look like
a single smooth hump, or something else? What might explain that shape in a delivery
business (hint: think about whether QuickCart might offer more than one type of delivery
service)?

*Your answer:*

At ~20 bins the distribution is clearly **bimodal** — a sharp early peak around 4–6
hours and a broader, longer-tailed peak around 15–25 hours. That's exactly what you'd
expect if QuickCart runs two different delivery products: a fast **express** track (the
narrow low peak) and a slower **standard** track (the broader higher peak). The
right-hand tail is even fatter than a single normal would produce, which hints that one
region is dragging the standard bucket further out.


## Task 2 — Density Plot & Bandwidth 

A histogram's bin width can hide or exaggerate structure. Let's check with a KDE curve,
where **bandwidth** plays the same role as bin width.

**Fill in the blank:** try `bw_adjust` values of `0.2`, `1.0`, and `2.5` (one at a time,
re-running the cell) and observe how the curve changes.

In [3]:
# 0.2 -> jagged, many spurious wiggles (over-fit).
# 1.0 -> smooth, two clear peaks: the honest picture.
# 2.5 -> over-smoothed into a single hump: hides the bimodality.
bw_adjust = 1.0

plt.figure()
sns.kdeplot(df["delivery_time_hours"], bw_adjust=bw_adjust, fill=True, color="#C44E52")
plt.title(f"Delivery Time Density (bw_adjust={bw_adjust})")
plt.xlabel("Delivery Time (hours)")
plt.ylabel("Density")
plt.show()


C:\Users\moksh\AppData\Local\Temp\ipykernel_11272\1375147708.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Question 2:** At which `bw_adjust` value do you clearly see **two peaks** instead of
one? What does each peak most likely represent in QuickCart's delivery operations?
What would happen if an analyst only ever used a large bandwidth here — what business
insight would they miss?

*Your answer:*

The two peaks appear most clearly at `bw_adjust=1.0`. The left peak (~5 h) is the
**express** service — fast, tightly clustered around its target time — and the right
peak (~18 h) is **standard** delivery. At `bw_adjust=2.5` the two peaks merge into a
single wide hump, and the analyst would conclude that delivery is one messy process
with a mean around 14 h. They would completely miss the fact that QuickCart is
actually running two distinct services with two different SLAs, and they would set
alerts and staffing plans against an average that doesn't describe any real order.


## Task 3 — Proportions: Which region needs help? 

Now let's look at `delivery_status` (On Time / Delayed / Severely Delayed) across regions.

**Fill in the blank:** the code below builds the proportion table and three chart types.
Nothing to change here except reading the output carefully — but try commenting out one of
the three chart types (`axes[0]`, `axes[1]`, or `axes[2]`) to see how the layout responds.

In [4]:
status_by_region = (
    df.groupby("region")["delivery_status"]
    .value_counts(normalize=True)
    .unstack()
    .reindex(columns=["On Time", "Delayed", "Severely Delayed"])
)
status_by_region.round(2)

delivery_status,On Time,Delayed,Severely Delayed
region,,,
Central,0.35,0.41,0.24
East,0.33,0.51,0.16
North,0.38,0.45,0.17
South,0.04,0.42,0.54
West,0.35,0.45,0.20


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Pie chart -- one region only, since pie charts don't compare groups well
axes[0].pie(status_by_region.loc["South"], labels=status_by_region.columns,
            autopct="%1.0f%%", colors=["#55A868", "#DD8452", "#C44E52"])
axes[0].set_title("Pie Chart\n(South region only)")

# Stacked bar -- all regions
status_by_region.plot(kind="bar", stacked=True, ax=axes[1],
                       color=["#55A868", "#DD8452", "#C44E52"])
axes[1].set_title("Stacked Bar Chart\n(all regions)")
axes[1].set_ylabel("Proportion")
axes[1].legend(fontsize=8)

# Side-by-side bar -- all regions
status_by_region.plot(kind="bar", stacked=False, ax=axes[2],
                       color=["#55A868", "#DD8452", "#C44E52"])
axes[2].set_title("Side-by-side Bar Chart\n(all regions)")
axes[2].set_ylabel("Proportion")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

C:\Users\moksh\AppData\Local\Temp\ipykernel_11272\1276047800.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Question 3:** Which region has the worst delivery performance, and which chart made
that easiest to spot? Why would the pie chart alone (even one per region) be a poor
choice for comparing all five regions against each other?

*Your answer:*

**South** has the worst performance — its "Severely Delayed" slice is far larger than
any other region's, and its "On Time" share is the smallest. The **side-by-side bar
chart** made that easiest to see: the "Severely Delayed" bars sit on a shared baseline,
so comparing their heights is a direct length comparison. The stacked bar shows the
same story but you have to eyeball the length of a middle segment, which is harder.
Five separate pie charts would be a poor choice because comparing angles across
different pies is far less accurate than comparing lengths on a shared axis — the pie
chart in the figure only makes sense here because it shows a single region.


## Wrap-up — Your Recommendation *(~3 min)*

Write a 2–3 sentence recommendation to the QuickCart ops team, combining **both** findings:
- What Task 1 & 2 revealed about delivery time structure (histogram + KDE + bandwidth)
- What Task 3 revealed about regional performance (proportions)

*Your recommendation:*

Delivery time isn't one process — it's two: an express service tightly centred around
~5 hours and a standard service centred around ~18 hours, and any single "average
delivery time" metric will mislead planning. On top of that, the **South region** is
dragging the standard bucket well beyond the SLA, with a much larger "Severely
Delayed" share than any other region. Priority for extra delivery staff should go to
South first, and internal reporting should split KPIs by express vs. standard rather
than reporting an overall mean.
